In [1]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

loader = PyMuPDFLoader("/Users/apple/Medical-Chatbot-using-LangChain-LLM-Pinecone-Flask-AWS/data/Medical_book.pdf")
docs = loader.load()
print(f"Number of documents: {len(docs)}")
for i in range(5):
    print(f"\nPage {i}")
    print(len(docs[i].page_content))
    print(docs[i].page_content[:200])

/Users/apple/Medical-Chatbot-using-LangChain-LLM-Pinecone-Flask-AWS/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/apple/Medical-Chatbot-using-LangChain-LLM-Pinecone-Flask-AWS/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of documents: 637

Page 0
0


Page 1
48
The GALE
ENCYCLOPEDIA
of MEDICINE
SECOND EDITION

Page 2
192
The GALE
ENCYCLOPEDIA
of MEDICINE
SECOND EDITION
J A C Q U E L I N E  L .  L O N G E ,  E D I T O R
D E I R D R E  S .  B L A N C H F I E L D ,  A S S O C I AT E  E D I T O R
V O L U M E
A-B
1

Page 3
3817
STAFF
Jacqueline L. Longe, Project Editor
Deirdre S. Blanchfield, Associate Editor
Christine B. Jeryan, Managing Editor
Donna Olendorf, Senior Editor
Stacey Blachford, Associate Editor
Kate Kretschman

Page 4
670
Introduction....................................................ix
Advisory Board..............................................xi
Contributors .................................................xiii
Ent


In [2]:
clean_docs = [doc for doc in docs if doc.page_content.strip() != ""]

In [3]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [4]:
minimal_docs = filter_to_minimal_docs(clean_docs)
print("Total minimal docs:", len(minimal_docs))

for i in range(3):
    print("\nDoc", i)
    print("Length:", len(minimal_docs[i].page_content))
    print(minimal_docs[i].page_content[:300])

Total minimal docs: 636

Doc 0
Length: 48
The GALE
ENCYCLOPEDIA
of MEDICINE
SECOND EDITION

Doc 1
Length: 192
The GALE
ENCYCLOPEDIA
of MEDICINE
SECOND EDITION
J A C Q U E L I N E  L .  L O N G E ,  E D I T O R
D E I R D R E  S .  B L A N C H F I E L D ,  A S S O C I AT E  E D I T O R
V O L U M E
A-B
1

Doc 2
Length: 3817
STAFF
Jacqueline L. Longe, Project Editor
Deirdre S. Blanchfield, Associate Editor
Christine B. Jeryan, Managing Editor
Donna Olendorf, Senior Editor
Stacey Blachford, Associate Editor
Kate Kretschmann, Melissa C. McDade, Ryan
Thomason, Assistant Editors
Mark Springer, Technical Specialist
Andrea Lo


In [5]:
# now we will do text splitting ( dividing the big doc into smaller chunks)
from langchain.text_splitter import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20) # means 500 characters ka group banega aur 20 characters ka overlap hoga in every 500 characters chunk
chunks = splitter.split_documents(minimal_docs)
print("Total chunks:", len(chunks))

for i in range(3):
    print("\nChunk", i)
    print("Length:", len(chunks[i].page_content))
    print(chunks[i].page_content[:200])

Total chunks: 5777

Chunk 0
Length: 48
The GALE
ENCYCLOPEDIA
of MEDICINE
SECOND EDITION

Chunk 1
Length: 192
The GALE
ENCYCLOPEDIA
of MEDICINE
SECOND EDITION
J A C Q U E L I N E  L .  L O N G E ,  E D I T O R
D E I R D R E  S .  B L A N C H F I E L D ,  A S S O C I AT E  E D I T O R
V O L U M E
A-B
1

Chunk 2
Length: 492
STAFF
Jacqueline L. Longe, Project Editor
Deirdre S. Blanchfield, Associate Editor
Christine B. Jeryan, Managing Editor
Donna Olendorf, Senior Editor
Stacey Blachford, Associate Editor
Kate Kretschman


In [6]:
!pip uninstall torch numpy sentence-transformers -y
!pip install numpy==1.26.4
!pip install torch
!pip install sentence-transformers

Found existing installation: torch 2.2.2
Uninstalling torch-2.2.2:
  Successfully uninstalled torch-2.2.2
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: sentence-transformers 5.1.2
Uninstalling sentence-transformers-5.1.2:
  Successfully uninstalled sentence-transformers-5.1.2
  Using cached numpy-1.26.4-cp39-cp39-macosx_10_9_x86_64.whl (20.6 MB)
You should consider upgrading via the '/Users/apple/Medical-Chatbot-using-LangChain-LLM-Pinecone-Flask-AWS/venv/bin/python3 -m pip install --upgrade pip' command.
  Using cached torch-2.2.2-cp39-none-macosx_10_9_x86_64.whl (150.8 MB)
You should consider upgrading via the '/Users/apple/Medical-Chatbot-using-LangChain-LLM-Pinecone-Flask-AWS/venv/bin/python3 -m pip install --upgrade pip' command.
  Using cached sentence_transformers-5.1.2-py3-none-any.whl (488 kB)
You should consider upgrading via the '/Users/apple/Medical-Chatbot-using-LangChain-LLM-Pineco

In [13]:
# now we will do vector embeddings and store it in Faiss vector database
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
texts = [chunk.page_content for chunk in chunks]
vectorstore = FAISS.from_texts(texts, embedding_model)

vectorstore.save_local("faiss_index")


# for optimzation we can save the index locally and load it when needed instead of recomputing the embeddings every time.
vectorstore = FAISS.load_local("faiss_index", embedding_model,allow_dangerous_deserialization=True)

In [14]:

# Now we will add our retriever
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3}) # k means return top 3 similar result 

In [17]:
# now we will ad our LLM ( ollama)
from langchain_ollama import OllamaLLM
llm = OllamaLLM(model="phi3")

In [22]:
# create the RAG chain
from langchain.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)

In [26]:
# testing the chat bot
from IPython.display import display, Markdown

query = "What are the black spots on face and how to fix it?"
response = qa_chain.invoke(query)
display(Markdown(response["result"]))

Black spots or dark patches on your skin, often referred to as age spots or liver spots, can result from excessive sun exposure. These appear when melanin in our skin becomes concentrated over time due to UV damage, causing discoloration and pigmentation changes that create these black marks.

Here are some ways you might help reduce the appearance of dark spots on your face:

1. **Sun Protection** - The first step is always sun protection because harmful ultraviolet (UV) rays can worsen existing skin conditions and cause new ones, including age or liver spots. Always wear a broad-spectrum sunscreen with an SPF of at least 30 on exposed areas even when it's cloudy outside as UVA/B rays can penetrate through clouds too.
   
2. **Topical Treatments** - Over-the-counter products such as hydroquinone, which lightens dark skin patches by slowing the production of melanin in your skin cells and promoting cell turnover to reveal lighter layers of skin beneath it; retinoids like tretinoin that help unblock pores but can cause irritation especially at initiation.
   
3. **Prescription Treatments** - A dermatologist might prescribe oral medications, such as Isotretinoin for severe acne-prone skin with blackheads; however, these should be used under medical supervision due to the potential side effects and contraindications of this powerful drug.
   
4. **Dermabrasion** - This is a professional treatment where your doctor removes layers of top skin cells using wire brushes or diamond-tipped devices over infected areas, removing discolored spots effectively for some patients but it can cause irritation and scarring in rare cases so should be done by professionals.
   
5. **Chemical Peels** - This method uses a chemical solution to remove the outer layer of skin that contains pigment-producing cells, revealing fresher layers underne0beneath this old or damaged skin but again it can cause irritation and scarring if overdone so should only be done by professionals.
   
6. **Microdermabrasion** - A procedure in which the outermost layer of your own dead cells are removed using a fine crystal-based tool, while simultaneously being gently massaged into healthy new skin and it is usually safe for most people when done correctly but should not be used as regular exfoliation.
   
7. **Laser Treatments** - Laser treatments can target melanin in the affected area to lighten dark spots, however this method might require multiple sessions with professional help and it's best to discuss about risks/benefits of this option under a dermatologist or cosmetic doctor.
   
8. **Home Remedies** - Some people find temporary relief from applying lemon juice due to its natural citric acid content that helps lighten dark spots but can cause further skin damage if not used properly and always diluted, so caution is advised when considering this method at home without medical supervision.
   
9. **Dietary Adjustments** - Some studies suggest certain diets high in antioxidants or omega-3 fatty acids might help improve skin condition but more scientific evidence needed to conclusively link them with reduction of dark spots, so while maintaining a balanced healthy diet and consultation is advised.
   
10. **Cosmetic Makeup** - You can use makeup specifically designed for covering blemishes or even tattoo color correctors underneath foundation that matches your skin tone to camouflage dark spots if they bother you, but remember it's just a temporary solution and doesn’t treat the underlying problem.
   
Please consult with a dermatologist before starting any treatment for them best advice tailored on individual condition of yours should be obtained from professional medical help as each case is unique so don't try anything without proper guidance because skin conditions can get worse if not treated correctly and some self-help methods might also bring unwanted side effects.